# Acquiring data from APIs
We use the **World Bank Indicators API**: free, public and no key required.
Documentation: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392

Each section matches a slide.

## JSON text to Python

In [ ]:
import json

raw = '{"iso3": "KEN", "gdp": [1.2, 1.3]}'
data = json.loads(raw)
print(type(data))
print(data["iso3"], data["gdp"][0])
print(json.dumps(data, indent=2))

## A nested World Bank record

In [ ]:
text = '''{"country": {"id": "KE",
 "value": "Kenya"}, "date": "2023",
 "countryiso3code": "KEN", "value": 16.7}'''
record = json.loads(text)
print(record["country"])
print(record["country"]["value"])
print(record["date"], record["value"])

## Sending a request

In [ ]:
import requests

BASE = "https://api.worldbank.org/v2"
INDICATOR = "NE.EXP.GNFS.ZS"  # exports % GDP
path = f"country/KEN/indicator/{INDICATOR}"
params = {"format": "json",
          "date": "2015:2023"}
resp = requests.get(f"{BASE}/{path}",
                    params=params,
                    timeout=60)
print(resp.status_code)
print(resp.url)

## The response object

In [ ]:
print(resp.status_code)
print(resp.headers["Content-Type"])
print(len(resp.text), "characters")
print(resp.text[:40])

## Unpacking the response

In [ ]:
payload = resp.json()
print(type(payload), len(payload))
meta, records = payload
print(meta["total"], "records in total")
print(meta["pages"], "page(s)")

## Looping over records

In [ ]:
for r in records[:5]:
    print(r["date"], r["value"])

## From records to a DataFrame

In [ ]:
import pandas as pd

rows = [
    {"iso3": r["countryiso3code"],
     "year": int(r["date"]),
     "value": r["value"]}
    for r in records
]
df = pd.DataFrame(rows).sort_values("year")
df.tail()

## A reusable function

In [ ]:
def fetch_indicator(countries, indicator,
                    start, end):
    codes = ";".join(countries)
    url = f"{BASE}/country/{codes}/indicator/"
    params = {"format": "json",
              "date": f"{start}:{end}",
              "per_page": 1000}
    resp = requests.get(url + indicator,
                        params, timeout=60)
    resp.raise_for_status()
    payload = resp.json()
    if len(payload) < 2 or not payload[1]:
        raise ValueError(payload)
    return payload[1]

## Calling the function

In [ ]:
recs = fetch_indicator(["KEN", "NGA", "ZAF"],
                       "NY.GDP.MKTP.CD",
                       2019, 2023)
print(len(recs), "records")
print(recs[0]["country"]["value"],
      recs[0]["date"])

## 200 OK is not always success

In [ ]:
bad = requests.get(f"{BASE}/country/XXX/"
                   "indicator/NY.GDP.MKTP.CD",
                   params={"format": "json"},
                   timeout=60)
print(bad.status_code)
print(bad.json()[0]["message"][0]["value"])

## What a timeout looks like

In [ ]:
try:
    requests.get(BASE, timeout=0.001)
except requests.Timeout as e:
    print("Timed out:", type(e).__name__)
except requests.RequestException as e:
    print("Network error:", type(e).__name__)

## Loading a cached copy

In [ ]:
from pathlib import Path

CACHE = Path("../../data/cache")

def load_cached(name):
    path = CACHE / name
    with open(path, encoding="utf-8") as f:
        return json.load(f)[1]

print(len(load_cached("wb_gdp_usd.json")))

## Falling back to the cache

In [ ]:
try:
    recs = fetch_indicator(
        ["KEN"], INDICATOR, 2015, 2023)
    print("From the API")
except requests.RequestException as e:
    print("API unavailable, using cache:", e)
    recs = load_cached(
        "wb_exports_pct_gdp.json")
print(len(recs), "records")